In [ ]:
!pip install thop

In [ ]:
# 1. Install dependency
!pip install thop -q

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import time
import pandas as pd
from thop import profile

# --- GPU CHECK ---
if not torch.cuda.is_available():
    raise SystemError("GPU not found! Please go to Runtime > Change runtime type and select 'T4 GPU'.")

# --- CONFIGURATION ---
BATCH_SIZE = 128
EPOCHS = 1  # Full epoch to get meaningful accuracy
MODELS = ["resnet18", "resnet50"]
OPTIMIZERS = ["Adam", "SGD"]
device = torch.device('cuda')

# --- PREPARE DATA ---
transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_set = torchvision.datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
train_loader = torch.utils.data.DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
test_set = torchvision.datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)
test_loader = torch.utils.data.DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False)

def get_model(model_name):
    if model_name == "resnet18":
        model = torchvision.models.resnet18(weights=None)
    else:
        model = torchvision.models.resnet50(weights=None)
    model.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
    model.fc = nn.Linear(model.fc.in_features, 10) # 10 classes for FashionMNIST
    return model

# --- EXPERIMENT ENGINE ---
results = []

print(f"Executing benchmarks on: {torch.cuda.get_device_name(0)}\n")

for model_name in MODELS:
    # Calculate FLOPs
    model_temp = get_model(model_name)
    flops, params = profile(model_temp, inputs=(torch.randn(1, 1, 32, 32),), verbose=False)

    for opt_name in OPTIMIZERS:
        model = get_model(model_name).to(device)
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(model.parameters(), lr=0.001) if opt_name == "Adam" else optim.SGD(model.parameters(), lr=0.01, momentum=0.9)

        # Warmup
        dummy_input = torch.randn(2, 1, 32, 32).to(device)
        model(dummy_input)
        torch.cuda.synchronize()

        # --- Training Phase ---
        print(f"Training {model_name.upper()} with {opt_name}...")
        start_time = time.time()
        model.train()
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

        torch.cuda.synchronize()
        total_time = time.time() - start_time

        # --- Evaluation Phase (Accuracy) ---
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for images, labels in test_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        accuracy = 100 * correct / total

        results.append({
            "Model": model_name.upper(),
            "Optimizer": opt_name,
            "GFLOPs": round(flops / 1e9, 4),
            "Params(M)": round(params / 1e6, 2),
            "Time(s)": round(total_time, 2),
            "Accuracy(%)": round(accuracy, 2)
        })

# --- FINAL OUTPUT ---
df = pd.DataFrame(results)
print("\n" + "="*80)
print("FINAL GPU REPORT: 4 ROWS | 6 COLUMNS")
print("="*80)
print(df.to_string(index=False))

100%|██████████| 26.4M/26.4M [00:02<00:00, 9.70MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 179kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 3.57MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 29.4MB/s]


Executing benchmarks on: Tesla T4

Training RESNET18 with Adam...
Training RESNET18 with SGD...
Training RESNET50 with Adam...
Training RESNET50 with SGD...

FINAL GPU REPORT: 4 ROWS | 6 COLUMNS
   Model Optimizer  GFLOPs  Params(M)  Time(s)  Accuracy(%)
RESNET18      Adam  0.0356      11.18    22.93        87.27
RESNET18       SGD  0.0356      11.18    22.07        86.35
RESNET50      Adam  0.0827      23.52    47.62        84.25
RESNET50       SGD  0.0827      23.52    44.83        83.33
